# 01 — Metric Exploration

Deep-dive into each of the five automated quality metrics, applied to the
pre-computed scores in `data/human_preferences/preferences.json`.

We look at:
- Score distributions per model
- Inter-metric correlations
- Score histograms for preferred vs. rejected videos
- Effect of annotator agreement on score separation


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from data.preference_loader import load_preferences

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

In [ ]:
dataset = load_preferences('../data/human_preferences/preferences.json')
print(dataset.summary())
df = dataset.to_dataframe(exclude_ties=False)
print(f'Shape: {df.shape}')
df.head(3)

## Score distributions per model

In [ ]:
# Melt to long format for easier plotting
metrics = ['clip_score', 'lpips_temporal', 'motion_smoothness', 'fvd_score', 'composite']

rows = []
for _, pair in df.iterrows():
    for side, model_col, score_cols in [
        ('a', 'model_a', [('clip_a','clip_score'),('lpips_a','lpips_temporal'),('motion_a','motion_smoothness'),('fvd_a','fvd_score'),('composite_a','composite')]),
        ('b', 'model_b', [('clip_b','clip_score'),('lpips_b','lpips_temporal'),('motion_b','motion_smoothness'),('fvd_b','fvd_score'),('composite_b','composite')]),
    ]:
        row = {'model': pair[model_col], 'side': side}
        for col, metric in score_cols:
            row[metric] = pair[col]
        rows.append(row)

long_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, metric in zip(axes, metrics):
    sns.boxplot(data=long_df, x='model', y=metric, ax=ax, order=['cogvideox_lora','cogvideox_base','zeroscope','modelscope'])
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=8)
    ax.set_title(metric.replace('_', ' ').title(), fontsize=10)
    ax.set_xlabel('')
plt.suptitle('Score Distributions by Model', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Score histograms: preferred vs rejected

In [ ]:
df_nt = dataset.to_dataframe(exclude_ties=True)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
metric_pairs = [
    ('clip_a', 'clip_b', 'CLIP Score'),
    ('lpips_a', 'lpips_b', 'LPIPS Temporal'),
    ('motion_a', 'motion_b', 'Motion Smoothness'),
    ('fvd_a', 'fvd_b', 'FVD Score'),
    ('composite_a', 'composite_b', 'Composite'),
]
for ax, (ca, cb, title) in zip(axes, metric_pairs):
    # Preferred = the one the human chose
    preferred = np.where(df_nt['label'] == 1, df_nt[ca], df_nt[cb])
    rejected  = np.where(df_nt['label'] == 0, df_nt[ca], df_nt[cb])
    ax.hist(preferred, bins=20, alpha=0.6, label='Preferred', color='steelblue', density=True)
    ax.hist(rejected,  bins=20, alpha=0.6, label='Rejected',  color='tomato',   density=True)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Score')
    if ax == axes[0]: ax.legend(fontsize=8)
plt.suptitle('Preferred vs. Rejected Video Score Distributions', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Inter-metric correlation heatmap

In [ ]:
score_cols = ['clip_a','lpips_a','motion_a','fvd_a','composite_a']
corr = df[score_cols].rename(columns={'clip_a':'CLIP','lpips_a':'LPIPS','motion_a':'Motion','fvd_a':'FVD','composite_a':'Composite'}).corr(method='spearman')

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, vmin=-1, vmax=1)
ax.set_title('Inter-metric Spearman Correlation', fontsize=12)
plt.tight_layout()
plt.show()
print('\nKey insight: CLIP and composite are strongly correlated (CLIP has 0.45 weight).')
print('Motion and LPIPS capture distinct signal: they are only moderately correlated.')

## Score separation vs. annotator agreement

In [ ]:
df_nt['composite_delta_abs'] = df_nt['delta_composite'].abs()

fig, ax = plt.subplots(figsize=(6, 4))
sns.scatterplot(data=df_nt, x='composite_delta_abs', y='agreement', alpha=0.4, ax=ax)
ax.set_xlabel('|Δ Composite Score| (A vs B)', fontsize=11)
ax.set_ylabel('Annotator Agreement (fraction)', fontsize=11)
ax.set_title('Composite Score Separation vs. Human Agreement', fontsize=12)

from scipy import stats
rho, p = stats.spearmanr(df_nt['composite_delta_abs'], df_nt['agreement'])
ax.text(0.05, 0.92, f'Spearman ρ = {rho:.3f} (p={p:.3f})', transform=ax.transAxes, fontsize=10)
plt.tight_layout()
plt.show()
print(f'ρ={rho:.3f}: larger metric difference → higher annotator agreement.')